# Pommerman FFA Baselines

Train and evaluate practical baseline policies for full four-agent Pommerman FFA. The defaults are smoke-test sized; increase `N_EPISODES` for meaningful learning curves.


In [ ]:
from pathlib import Path
import sys


def _find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "discrete_action_space").exists() and (path / "relevant_papers").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {current}")

ROOT = _find_repo_root()
for path in (ROOT, ROOT / "discrete_action_space", ROOT / "discrete_action_space" / "bimatrix_game"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from discrete_action_space.pommerman_ffa.notebook_utils import (
    evaluate_policy,
    evaluate_random_reference,
    evaluate_simple_agent_reference,
    plot_evaluation_rewards,
    plot_training_curves,
    policy_from_iql,
    policy_from_ippo,
    train_iql_dqn,
    train_ippo,
)

POMMERMAN_DIR = ROOT / "discrete_action_space" / "pommerman_ffa"

N_EPISODES = 100
MAX_STEPS = 200
EVAL_EPISODES = 20
SEED = 2025
USE_GPU = True
OUTPUT_ROOT = POMMERMAN_DIR / "runs" / "baselines"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Random Reference


In [ ]:
random_eval = evaluate_random_reference(
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 1000,
    output_dir=OUTPUT_ROOT / "random",
)
plot_evaluation_rewards(random_eval)


## Built-In SimpleAgent Reference


In [ ]:
simple_eval = evaluate_simple_agent_reference(
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 2000,
)
plot_evaluation_rewards(simple_eval)


## Shared-Parameter IQL/DQN


In [ ]:
iql_stats = train_iql_dqn(
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
)
plot_training_curves(iql_stats)

iql_eval = evaluate_policy(
    policy_from_iql(iql_stats["agent"]),
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 3000,
    output_dir=OUTPUT_ROOT / "iql_dqn",
    label="iql_dqn",
)
plot_evaluation_rewards(iql_eval)


## Shared-Parameter IPPO/PPO


In [ ]:
ippo_stats = train_ippo(
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 10,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
)
plot_training_curves(ippo_stats)

ippo_eval = evaluate_policy(
    policy_from_ippo(ippo_stats["agent"]),
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 4000,
    output_dir=OUTPUT_ROOT / "ippo",
    label="ippo",
)
plot_evaluation_rewards(ippo_eval)
